# Evidencia de infraestructura

Notebook de evidencia para el apartado de resultados de infraestructura del TFG. Ejecuta la validación contractual del stack y documenta el estado estructural de la base de datos.

- Validación programática reutilizando `scripts/validate_infra_stack.py`.
- Inventario de esquemas, tablas, hypertables y políticas de compresión.
- Exportación de evidencia fechada en JSON y CSV bajo `reports/validation/infra/`.

## Uso recomendado

**Prerrequisitos:** servicios Docker en `healthy`; ejecutar desde JupyterLab en el servicio `jupyter` (`Kernel > Restart Kernel and Run All`).

- Usa `00_infraestructura/01_smoke_test_infraestructura.ipynb` para una comprobación rápida de operatividad mínima antes de este notebook.
- Usa este notebook cuando necesites confirmar el contrato completo de infraestructura y guardar evidencia para seguimiento técnico.
- **Orden sugerido del pilar:** `01` (smoke) -> `02` (este notebook) -> opcionalmente `00_infraestructura/03_smoke_api_externa.ipynb` (conectividad HTTPS externa, opt-in `RUN_EXTERNAL_SMOKE=1`).

In [1]:
from __future__ import annotations

import json
import platform
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

# Silenciar warnings
warnings.filterwarnings(
    "ignore",
    message=".*pkg_resources is deprecated.*",
    category=UserWarning,
)

# Resuelve raíz del proyecto buscando scripts/validate_infra_stack.py
ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "scripts" / "validate_infra_stack.py").exists():
        ROOT = candidate
        break

assert ROOT is not None, "No se encontró la raíz del proyecto (falta scripts/validate_infra_stack.py)."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.validate_infra_stack import CHECK_LABELS, InfraValidator

run_meta = {
    "fecha_hora_utc": datetime.now(timezone.utc).isoformat(),
    "raíz_proyecto": str(ROOT),
    "python": platform.python_version(),
    "plataforma": platform.platform(),
}
run_meta

{'fecha_hora_utc': '2026-06-03T12:23:59.637360+00:00',
 'raíz_proyecto': '/app',
 'python': '3.10.19',
 'plataforma': 'Linux-5.15.133.1-microsoft-standard-WSL2-x86_64-with-glibc2.36'}

In [2]:
validator = InfraValidator(allow_missing_mlflow_uri=False)
exit_code = validator.run()

records = [
    {
        "check_id": rec.check_id,
        "label": CHECK_LABELS.get(rec.check_id, rec.check_id),
        "status": rec.status,
        "required": rec.required,
        "detail": rec.detail,
    }
    for rec in validator.records
]

df_raw = pd.DataFrame(records)
summary_df = pd.DataFrame(
    {
        "id_check": df_raw["check_id"],
        "etiqueta": df_raw["label"],
        "estado": df_raw["status"],
        "requerido": df_raw["required"].map({True: "Sí", False: "No"}),
        "detalle": df_raw["detail"],
    }
)
summary_df

2026-06-03 12:23:59 [info     ] VALIDACION_INFRA_INICIADA      allow_missing_mlflow_uri=False job=infra_stack_validation
2026-06-03 12:23:59 [info     ] motor_bd_creando               componente=base_datos dsn=postgresql+psycopg2://tfg_admin:***@db:5432/tfg_quant_db max_sobrecupo=10 tamano_pool=5 timeout_conexion=10
2026-06-03 12:24:00 [info     ] motor_bd_creado                componente=base_datos dsn=postgresql+psycopg2://tfg_admin:***@db:5432/tfg_quant_db
2026-06-03 12:24:00 [info     ] conexion_bd_ok                 componente=base_datos
2026-06-03 12:24:00 [debug    ] version_bd_obtenida            componente=base_datos version=15.13
2026-06-03 12:24:00 [info     ] motor_bd_creando               componente=base_datos dsn=postgresql+psycopg2://tfg_admin:***@db:5432/tfg_quant_db max_sobrecupo=10 tamano_pool=5 timeout_conexion=10
2026-06-03 12:24:00 [info     ] motor_bd_creado                componente=base_datos dsn=postgresql+psycopg2://tfg_admin:***@db:5432/tfg_quant_db
2026-06-0

,id_check,etiqueta,estado,requerido,detalle
0,DB_CONNECTION,Conexión PostgreSQL,PASS,Sí,Conectado a PostgreSQL 15.13.
1,TIMESCALE_EXTENSION,Extensión Timescale instalada,PASS,Sí,Versión de la extensión timescaledb=2.23.1
2,SCHEMAS_READY,Esquemas críticos disponibles,PASS,Sí,Todos los esquemas críticos presentes (4).
3,TABLES_READY,Tablas críticas disponibles,PASS,Sí,Todas las tablas críticas presentes (12/12).
4,FEATURE_STORE_READY,Contrato mínimo de feature_store,PASS,Sí,El esquema feature_store y la tabla pipeline_r...
5,MLFLOW_CONNECTION,Conectividad de tracking MLflow,PASS,Sí,Conectado a tracking_uri=http://mlflow:5000
6,MLFLOW_WRITE,Escritura mínima en MLflow,PASS,Sí,Run efímero creado y finalizado (run_id=3ae496...


## Evidencia estructural de la base de datos

Tras superar la validación contractual, se documenta el estado estructural real de la base de datos: versiones del motor, esquemas creados por `docker/init.sql`, inventario de tablas por dominio y configuración de TimescaleDB (hypertables y políticas de compresión).

In [3]:
from src.utils.database import create_db_engine
from sqlalchemy import text

# Inicializa motor de evidencia para consultas estructurales de infraestructura
evidence_engine = create_db_engine()

with evidence_engine.connect() as conn:
    pg_version = conn.execute(text("SHOW server_version")).scalar()
    ts_row = conn.execute(text(
        "SELECT extversion FROM pg_extension WHERE extname = 'timescaledb'"
    )).fetchone()
    ts_version = ts_row[0] if ts_row else "no instalada"

version_info = {"PostgreSQL": pg_version, "TimescaleDB": ts_version}

version_df = pd.DataFrame(
    [{"Componente": k, "Versión": v} for k, v in version_info.items()]
)
version_df

2026-06-03 12:24:07 [info     ] motor_bd_creando               componente=base_datos dsn=postgresql+psycopg2://tfg_admin:***@db:5432/tfg_quant_db max_sobrecupo=10 tamano_pool=5 timeout_conexion=10
2026-06-03 12:24:07 [info     ] motor_bd_creado                componente=base_datos dsn=postgresql+psycopg2://tfg_admin:***@db:5432/tfg_quant_db


,Componente,Versión
0,PostgreSQL,15.13
1,TimescaleDB,2.23.1


In [4]:
PROJECT_SCHEMAS = [
    "market_data", "microstructure_data", "onchain_data",
    "derivatives_data", "alternative_data", "feature_store",
    "trading", "mlflow_meta",
]

with evidence_engine.connect() as conn:
    tables_df = pd.read_sql(text(
        "SELECT table_schema, table_name "
        "FROM information_schema.tables "
        "WHERE table_schema = ANY(:schemas) "
        "ORDER BY table_schema, table_name"
    ), conn, params={"schemas": PROJECT_SCHEMAS})

tables_summary = (
    tables_df.groupby("table_schema")
    .size()
    .rename("tablas")
    .reset_index()
    .rename(columns={"table_schema": "esquema"})
    .sort_values("esquema")
)
print(f"{len(tables_df)} tablas en {len(tables_summary)} esquemas del proyecto\n")
tables_summary

49 tablas en 8 esquemas del proyecto



,esquema,tablas
0,alternative_data,7
1,derivatives_data,5
2,feature_store,2
3,market_data,3
4,microstructure_data,7
5,mlflow_meta,16
6,onchain_data,8
7,trading,1


In [5]:
with evidence_engine.connect() as conn:
    ht_df = pd.read_sql(text(
        "SELECT hypertable_schema AS esquema, "
        "hypertable_name AS hypertable, "
        "compression_enabled AS compresion, "
        "num_chunks AS chunks "
        "FROM timescaledb_information.hypertables"
    ), conn).rename(columns={"compresion": "compresión"})

    comp_df = pd.read_sql(text(
        "SELECT hypertable_schema AS esquema, "
        "hypertable_name AS hypertable, "
        "schedule_interval::text AS intervalo_politica "
        "FROM timescaledb_information.jobs "
        "WHERE proc_name = 'policy_compression'"
    ), conn).rename(columns={"intervalo_politica": "intervalo_política"})

evidence_engine.dispose()

print("Hypertables registradas:")
display(ht_df)
print(f"\nPolíticas de compresión: {len(comp_df)} activa(s)")
if not comp_df.empty:
    display(comp_df)

Hypertables registradas:


,esquema,hypertable,compresión,chunks
0,market_data,market_ohlcv,True,459



Políticas de compresión: 1 activa(s)


,esquema,hypertable,intervalo_política
0,market_data,market_ohlcv,12:00:00


## Exportación y productos bajo `reports/`

El bloque de exportación persiste dos artefactos con sello UTC (`%Y%m%dT%H%M%SZ`) en el nombre:

| Producto | Ruta |
| --- | --- |
| Informe contractual (checks, versiones, inventario) | `reports/validation/infra/infra_contract_validation_{stamp}.json` |
| Resumen tabular de checks | `reports/validation/infra/infra_contract_validation_{stamp}.csv` |

No hay figuras (`savefig`) ni export a `docs/`. La comprobación mínima previa corresponde a `00_infraestructura/01_smoke_test_infraestructura.ipynb`.

In [6]:
# Exporta evidencia en JSON y CSV para trazabilidad
# Normaliza intervalos PostgreSQL que llegan como datetime.timedelta o pd.Timedelta
# Aplica sanitizador recursivo antes de json.dumps
import math
from datetime import date, timedelta
from decimal import Decimal

import numpy as np


def _json_sanitize(obj):  
    """Convierte estructuras anidadas a tipos compatibles con json.dumps."""
    if obj is None:
        return None
    if isinstance(obj, bool):
        return obj
    if isinstance(obj, (int, float)):
        if isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)):
            return None
        return obj
    if isinstance(obj, str):
        return obj
    if isinstance(obj, dict):
        return {str(k): _json_sanitize(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_json_sanitize(x) for x in obj]
    if isinstance(obj, (datetime, date)):
        return obj.isoformat()
    if isinstance(obj, (timedelta, pd.Timedelta)):
        return str(obj)
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    if isinstance(obj, Decimal):
        return str(obj)
    if isinstance(obj, (np.integer, np.floating, np.bool_)):
        val = obj.item()
        if isinstance(val, float) and (np.isnan(val) or np.isinf(val)):
            return None
        return val
    try:
        if pd.isna(obj):
            return None
    except (TypeError, ValueError):
        pass
    return str(obj)


output_dir = ROOT / "reports" / "validation" / "infra"
output_dir.mkdir(parents=True, exist_ok=True)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
json_path = output_dir / f"infra_contract_validation_{stamp}.json"
csv_path = output_dir / f"infra_contract_validation_{stamp}.csv"

payload = {
    "metadata": run_meta,
    "exit_code": exit_code,
    "final_result": "PASS" if exit_code == 0 else "FAIL",
    "checks": records,
    "versions": {k: str(v) for k, v in version_info.items()},
    "schemas_table_count": tables_summary.to_dict(orient="records"),
    "hypertables": ht_df.to_dict(orient="records"),
    "compression_policies": comp_df.to_dict(orient="records"),
}

safe_payload = _json_sanitize(payload)
json_path.write_text(
    json.dumps(safe_payload, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
summary_df.to_csv(csv_path, index=False)

print(f"Evidencia JSON: {json_path}")
print(f"Resumen CSV:    {csv_path}")

Evidencia JSON: /app/reports/validation/infra/infra_contract_validation_20260603T122409Z.json
Resumen CSV:    /app/reports/validation/infra/infra_contract_validation_20260603T122409Z.csv


In [7]:
# Aplica criterio de lectura rápida de resultado final
if exit_code == 0:
    print("CONTRATO DE INFRA: PASS")
    print("Todos los checks requeridos han pasado.")
else:
    print("CONTRATO DE INFRA: FAIL")
    print("Revisa summary_df y el campo 'detalle' para acciones correctivas.")

CONTRATO DE INFRA: PASS
Todos los checks requeridos han pasado.
